# 第2回 宿題（発展）: 用語集の Q&A を LoRA で学習し、学習前後と検索方式を比べる

演習ノートブック（`notebooks/lora_format_tuning.ipynb`）をもとに、FAQ の代わりに **講座の用語集**（`data/glossary.json`）から作った Q&A を学習させます。

**確かめること**

1. 学習前のモデルは、講座の用語集の定義（講座に固有の内容を含む）を答えられるか。
2. 学習後、学習データと同じ言い回しの質問（`exact`）、言い換え・別名・逆引きの質問（`paraphrase`）、用語集にない語の質問（`unknown`）で、それぞれどう変わるか。
3. 学習しないまま用語集を検索してプロンプトに入れる方式（セクション 6）と比べて、どちらが向くか。

**データ**（`lectures/02/homework/` にあります）

| ファイル | 内容 |
| --- | --- |
| `glossary_sft.jsonl` | データ A。`uv run python scripts/make_sft_dataset.py` の出力そのまま（58 語 × 2 言い回し = 116 件） |
| `glossary_sft_plus.jsonl` | データ B。A に、用語集にない 15 語を聞かれたら「用語集に載っていません」と答える負例 30 件を足したもの（`make_plus_dataset.py` で作成） |
| `glossary_eval.json` | 評価セット 34 問（exact 12 / paraphrase 12 / unknown 10）と正解の語句 |
| `glossary.json` | 用語集（リポジトリの `data/glossary.json`）。セクション 6 の検索で使う |

**実行のしかた**

- **Colab**: T4 GPU を選び、上の 4 ファイルを左のファイル一覧から `/content` にアップロードしてから、上から実行します。
- **ローカル**（CUDA の GPU）: `lectures/02/homework/` に Colab と同じ版を入れた venv を作り、このディレクトリで実行します（`README.md` 参照）。
- どちらのデータで学習するかは、セクション 1 の `train_file` で選びます（環境変数 `GLOSSARY_TRAIN` でも変えられます）。A と B を比べるときは、**セッションを再起動して**から上から実行し直します（学習後のモデルが残っているため）。

## 0. 環境の確認とライブラリの導入

Colab のときだけ、演習と同じ版のライブラリを入れます。ローカルでは venv に入れた版をそのまま使います。

In [1]:
import subprocess, sys
IN_COLAB = "google.colab" in sys.modules
try:
    gpu_info = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv"], capture_output=True, text=True).stdout
except FileNotFoundError:
    gpu_info = ""
print("Colab" if IN_COLAB else "ローカル", "|", gpu_info or "GPU が見つかりません")

ローカル | name, memory.total [MiB]
NVIDIA GeForce RTX 5070, 12227 MiB



In [2]:
# 演習ノートブックと同じ版（Colab の T4 で動作確認済み）
PINS = {"transformers": "5.18.0", "peft": "0.21.1", "trl": "1.14.2", "accelerate": "1.15.0", "datasets": "5.1.0"}
if IN_COLAB:
    subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "-q", "torchao"])   # peft 0.21 と非互換なので外す
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *[f"{k}=={v}" for k, v in PINS.items()]])

import transformers, peft, trl, accelerate, datasets, torch
VERSIONS = {m.__name__: m.__version__ for m in [transformers, peft, trl, accelerate, datasets, torch]}
print(VERSIONS)
mismatch = {k: VERSIONS[k] for k, v in PINS.items() if VERSIONS[k] != v}
assert not mismatch, f"版が PINS と違います: {mismatch}"

/home/sokos/workspace/codes/lecture-ai-engineering-2026/lectures/02/homework/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


{'transformers': '5.18.0', 'peft': '0.21.1', 'trl': '1.14.2', 'accelerate': '1.15.0', 'datasets': '5.1.0', 'torch': '2.11.0+cu130'}


## 1. 設定

演習からの変更点:

- `train_file`: 学習データのファイル（A: `glossary_sft.jsonl` / B: `glossary_sft_plus.jsonl`）。
- `num_train_epochs`: 20 → 10。データが 116 件（B は 146 件）あるので、1 エポックが約 15 ステップ（B は約 19 ステップ）になります。10 エポックで約 150 ステップです。
- `max_new_tokens`: 96 → 256。用語の定義文は FAQ の文より長いので、96 だと途中で切れます。
- `use_system_prompt`: 学習・評価の両方で、先頭に同じ system プロンプト（用語集アシスタントの役割と「用語集に載っていない語は推測で説明しない」という指示）を付けます。演習の `SYSTEM` と同じく、**指示はプロンプトで済ませてある**状態から始め、学習で何が変わるかを見ます。

In [3]:
from pathlib import Path
import os, json, time

CONFIG = {
    "base_model": "Qwen/Qwen3.5-0.8B",
    "revision": "2fc06364715b967f1860aea9cf38778875588b17",
    "seed": 42,
    "train_file": os.environ.get("GLOSSARY_TRAIN", "glossary_sft.jsonl"),   # A: glossary_sft.jsonl / B: glossary_sft_plus.jsonl
    "num_train_epochs": 10,
    "warmup_steps": 10,
    "lr": 2e-4,
    "lora_r": 16,
    "lora_alpha": 32,
    "max_seq_len": 512,
    "max_new_tokens": 256,
    "gen": {"do_sample": False},
    "use_system_prompt": True,
}
VARIANT = Path(CONFIG["train_file"]).stem   # 結果ファイルの置き場所をデータごとに分ける
DATA_DIR = Path("/content") if IN_COLAB else Path.cwd()
GLOSSARY_FILE = DATA_DIR / "glossary.json"
if not GLOSSARY_FILE.exists():   # ローカルではリポジトリの data/glossary.json を使う
    GLOSSARY_FILE = Path.cwd().parents[2] / "data" / "glossary.json"
WORK = (Path("/content/glossary_lab") if IN_COLAB else Path.cwd() / "out") / VARIANT
WORK.mkdir(parents=True, exist_ok=True)
print(json.dumps(CONFIG, ensure_ascii=False, indent=2))
print("データ:", DATA_DIR, "| 用語集:", GLOSSARY_FILE, "| 結果の保存先:", WORK)

{
  "base_model": "Qwen/Qwen3.5-0.8B",
  "revision": "2fc06364715b967f1860aea9cf38778875588b17",
  "seed": 42,
  "train_file": "glossary_sft.jsonl",
  "num_train_epochs": 10,
  "warmup_steps": 10,
  "lr": 0.0002,
  "lora_r": 16,
  "lora_alpha": 32,
  "max_seq_len": 512,
  "max_new_tokens": 256,
  "gen": {
    "do_sample": false
  },
  "use_system_prompt": true
}
データ: /home/sokos/workspace/codes/lecture-ai-engineering-2026/lectures/02/homework | 用語集: /home/sokos/workspace/codes/lecture-ai-engineering-2026/lectures/02/homework/glossary.json | 結果の保存先: /home/sokos/workspace/codes/lecture-ai-engineering-2026/lectures/02/homework/out/glossary_sft


## 2. 学習データと評価セットの読み込み

学習データの各行は `{"messages": [user, assistant]}` の形で、演習の `to_chat` と同じ chat 形式です。

評価セットの 3 種類:

| kind | 例 | 正解の条件 |
| --- | --- | --- |
| `exact` | 「LoRA」とは何ですか？ | 学習データと同じ質問。定義文の語句（例: 「低ランク」と「行列」）を含む |
| `paraphrase` | Low-Rank Adaptationはどんな手法ですか？ / 学習データ全体を1周学習することを何と呼びますか？ | 学習データに無い言い回し、別名、逆引き（説明から用語名を答える）。語句を含む |
| `unknown` | 「RAG」とは何ですか？ / 「トークンブリッジ」とは何ですか？ | 用語集に無い語（実在する語 7、架空の語 3）。「用語集に載っていません」の旨を答える |

`must_include` は「グループごとにどれか 1 語」を含めば合格です。語句の一致は目安なので、出力の原文も自分の目で読んでください。

**見るところ**: `paraphrase` と `unknown` の質問が学習データに入っていないことを、ここで確かめます（入っているとエラーになります）。

In [4]:
def read_jsonl(path):
    return [json.loads(line) for line in Path(path).read_text(encoding="utf-8").splitlines() if line.strip()]

train_rows = read_jsonl(DATA_DIR / CONFIG["train_file"])
EVAL_SET = json.loads((DATA_DIR / "glossary_eval.json").read_text(encoding="utf-8"))
EVAL_ITEMS, UNKNOWN_MARKERS = EVAL_SET["items"], EVAL_SET["unknown_markers"]

train_questions = {r["messages"][0]["content"] for r in train_rows}
leaked = [it["q"] for it in EVAL_ITEMS if it["kind"] != "exact" and it["q"] in train_questions]
assert not leaked, f"評価用の質問が学習データに入っています: {leaked}"
n_negative = sum("用語集に載っていません" in r["messages"][1]["content"] for r in train_rows)
print(f"学習データ {CONFIG['train_file']}: {len(train_rows)} 件（うち用語集にない語の負例 {n_negative} 件）")
print("評価セット:", {k: sum(it["kind"] == k for it in EVAL_ITEMS) for k in ["exact", "paraphrase", "unknown"]})
print("学習例の 1 件:", train_rows[0]["messages"])

学習データ glossary_sft.jsonl: 116 件（うち用語集にない語の負例 0 件）
評価セット: {'exact': 12, 'paraphrase': 12, 'unknown': 10}
学習例の 1 件: [{'role': 'user', 'content': '「AIエンジニアリング」とは何ですか？'}, {'role': 'assistant', 'content': '学習済みの基盤モデル（主にLLM）を部品として使い、実際に役立つアプリケーションやシステムを設計・構築・評価・運用する技術領域。モデルを一から作ることより、モデルの選択、データやツールとの接続、評価と改善のサイクルを回すことに重心がある。'}]


## 3. モデルの読み込みと学習前評価

演習と同じく量子化なし（32bit）で読み込みます。生成は貪欲生成で、学習前・学習後・検索方式のすべてで同じ条件にします。

**採点のしかた**（以降のセクションでも同じ関数を使います）

- `exact` / `paraphrase`: 「用語集に載っていません」の旨を答えず、かつ `must_include` の語句を含めば ○。
- `unknown`: 「用語集に載っていません」の旨（`unknown_markers` のどれか）を答えれば ○。一般知識で説明しても、用語集にない語を説明した時点で ×（根拠のない断定と同じ扱い）。

**見るところ**: 種類ごとの正解率と、出力の原文。学習前のモデルは一般的な語（LoRA、トークンなど）を自分の知識で説明できても、講座に固有の内容（既定モデル、宿題の 3 レベルなど）は答えられないはずです。

In [5]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

DEVICE = "cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu")
tok = AutoTokenizer.from_pretrained(CONFIG["base_model"], revision=CONFIG["revision"])
if tok.pad_token is None: tok.pad_token = tok.eos_token
model = AutoModelForCausalLM.from_pretrained(CONFIG["base_model"], revision=CONFIG["revision"], dtype=torch.float32).to(DEVICE)
model.config.use_cache = True
print("loaded:", CONFIG["base_model"], "| device:", DEVICE, "| 重みのメモリ(GB):", round(model.get_memory_footprint() / 1e9, 2))

SYSTEM = ("あなたは AIE2026 講座の用語集アシスタントです。用語の意味を、講座の用語集の定義に沿って答えてください。"
          "用語集に載っていない語については、推測で説明せず『用語集に載っていません』と答えてください。")

def add_system(messages):
    return ([{"role": "system", "content": SYSTEM}] if CONFIG["use_system_prompt"] else []) + messages

def build_messages(question, context=None):
    # context: セクション 6 の検索方式で、用語集から取り出した項目をプロンプトに入れるときに使う
    user = question if context is None else f"用語集の該当項目:\n{context}\n\n質問: {question}"
    return add_system([{"role": "user", "content": user}])

def generate(model, question, context=None):
    prompt = tok.apply_chat_template(build_messages(question, context), tokenize=False, add_generation_prompt=True, enable_thinking=False)
    inputs = tok(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=CONFIG["max_new_tokens"], pad_token_id=tok.pad_token_id, **CONFIG["gen"])
    return tok.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()

Loading weights:   0%|          | 0/320 [00:00<?, ?it/s]

Loading weights:   0%|          | 1/320 [00:00<00:38,  8.33it/s]

Loading weights:  79%|███████▉  | 254/320 [00:00<00:00, 1373.21it/s]

Loading weights: 100%|██████████| 320/320 [00:00<00:00, 1224.50it/s]

loaded: Qwen/Qwen3.5-0.8B | device: cuda | 重みのメモリ(GB): 3.01


In [6]:
import re
import pandas as pd
pd.set_option("display.max_colwidth", None)
KINDS = ["exact", "paraphrase", "unknown"]

def said_unknown(raw):
    return any(m in raw for m in UNKNOWN_MARKERS)

def judge(item, raw):
    if item["kind"] == "unknown":
        return said_unknown(raw)
    return not said_unknown(raw) and all(any(w in raw for w in group) for group in item["must_include"])

def mark(ok):
    return "○" if ok else "×"

def evaluate(model, label, retrieve=None):
    '''評価セット 34 問を生成して採点する。retrieve を渡すと、検索した用語集の項目をプロンプトに入れる'''
    rows, t0 = [], time.time()
    for it in EVAL_ITEMS:
        hits = retrieve(it["q"]) if retrieve else None
        context = None if hits is None else ("\n".join(f"- {t['term']}: {t['definition']}" for t in hits) or "（該当する項目はありません）")
        raw = generate(model, it["q"], context)
        row = {"kind": it["kind"], "q": it["q"], "raw": raw, "ok": judge(it, raw)}
        if hits is not None: row["retrieved"] = [t["id"] for t in hits]
        rows.append(row)
    summary = {"label": label, "n": len(rows), "overall": sum(r["ok"] for r in rows) / len(rows),
               **{k: sum(r["ok"] for r in rows if r["kind"] == k) / sum(r["kind"] == k for r in rows) for k in KINDS},
               "seconds": round(time.time() - t0, 1)}
    (WORK / f"eval_{label}.json").write_text(json.dumps({"summary": summary, "rows": rows}, ensure_ascii=False, indent=2), encoding="utf-8")
    print(json.dumps(summary, ensure_ascii=False))
    return summary, rows

before_summary, before_rows = evaluate(model, "before")
display(pd.DataFrame([{"kind": r["kind"], "質問": r["q"], "学習前の出力": r["raw"], "判定": mark(r["ok"])} for r in before_rows]))

[transformers] `causal_conv1d_fn` is falling back to its reference PyTorch implementation because `causal_conv1d` is not installed. This is correct but much slower; install `causal_conv1d` for the optimized kernel.


[transformers] `chunk_gated_delta_rule` is falling back to its reference PyTorch implementation because `flash-linear-attention` is not installed. This is correct but much slower; install `flash-linear-attention` for the optimized kernel.


[transformers] `causal_conv1d_update` is falling back to its reference PyTorch implementation because `causal_conv1d` is not installed. This is correct but much slower; install `causal_conv1d` for the optimized kernel.


[transformers] `fused_recurrent_gated_delta_rule` is falling back to its reference PyTorch implementation because `flash-linear-attention` is not installed. This is correct but much slower; install `flash-linear-attention` for the optimized kernel.


{"label": "before", "n": 34, "overall": 0.058823529411764705, "exact": 0.0, "paraphrase": 0.16666666666666666, "unknown": 0.0, "seconds": 124.6}


,kind,質問,学習前の出力,判定
0,exact,「LoRA」とは何ですか？,LoRA（Low-Rank Adaptation）は、大規模言語モデル（LLM）のトレーニングにおいて、**「低秩（Low-Rank）」なパラメータを学習する手法**です。\n\nこの用語は、2024 年 11 月 1 日に公開された AIE2026 講座の用語集に載っており、その定義は以下の通りです。\n\n**LoRA は、大規模言語モデルの学習において、低秩なパラメータを学習する手法です。**\n\nこの手法は、モデルの学習プロセスにおいて、以下の 3 つの要素を組み合わせることで実現されます：\n1. **低秩なパラメータ**：モデルの学習初期に学習したパラメータが、より多くの学習ステップで維持されるように、低秩な構造を持つように設計されます。\n2. **低秩な補間**：学習された低秩なパラメータを、より多くの学習ステップで維持するために、低秩な補間（補間パラメータ）を学習します。\n3. **低秩な学習**：学習された低秩な補間パラメータを、より多くの学習ステップで維持するために,×
1,exact,「トークン」とは何ですか？,「トークン」とは、語彙学や言語処理において、**「語彙の単位」**または**「語の最小単位」**を指します。\n\n具体的には、以下の 3 つの観点から定義されます：\n\n1. **語彙の単位**：\n 語彙の単位として、**「単語（Word）」**を指します。例えば、日本語の「猫」や英語の「cat」などが、トークンの例です。\n\n2. **語の最小単位**：\n 語彙の単位として、**「語素（Phoneme）」**を指します。例えば、日本語の「猫」は「カ」や「ト」の音素から構成され、その音素がトークンの例です。\n\n3. **語の最小単位**：\n 語彙の単位として、**「音素（Phonotactic unit）」**を指します。例えば、日本語の「猫」は「カ」や「ト」の音素から構成され、その音素がトークンの例です。\n\nつまり、「トークン」とは、**「単語」または「音素」**という語�,×
2,exact,「ハルシネーション」とは何ですか？,「ハルシネーション」とは、物理学や工学において、**「光の反射と伝播が乱れ、波長が変化し、波の振幅が減少する現象」**を指します。\n\nこの現象は、主に以下の 2 つの観点から説明されます。\n\n1. **光の反射と伝播の乱れ**\n 光は通常、反射（表面からの反射）と伝播（波が移動する）の両方を同時に起こします。ハルシネーションは、この 2 つの現象が互いに衝突し、波長が変化し、波の振幅が減少する現象を指します。\n\n2. **波の振幅の減少**\n 光の振幅が減少する現象は、**ハルシネーション**と呼ばれます。\n\nこの現象は、光の反射と伝播の乱れによって生じます。ハルシネーションは、光の反射と伝播の乱れによって生じます。\n\n**結論**\n「ハルシネーション」とは、**光の反射と伝播の乱れによって生じる、波長が変化し、波の振幅が減少する現象**です。,×
3,exact,「エポック」とは何ですか？,「エポック」とは、**「エポック」とは、プロジェクトや組織の進捗状況、目標達成状況、または重要なイベントを指す概念です。**\n\n具体的には、以下の 3 つの観点から定義されます：\n\n1. **プロジェクトの進捗**：プロジェクトがどの段階（例：初期、進行中、完了）に位置するかを示す。\n2. **目標達成状況**：プロジェクトの目標が達成されているか、遅れているかを示す。\n3. **重要なイベント**：プロジェクトに重要なイベント（例：プロジェクト開始、プロジェクト終了、重要な変更など）を指す。\n\nこの用語は、プロジェクト管理や組織の計画・実行において、現在の状況に基づいて進捗管理や目標達成の判断に用いられるため、プロジェクトの「現在時」を指す言葉です。,×
4,exact,「量子化」とは何ですか？,「量子化」とは、物理学や情報科学において、**「量子状態（量子状態）」を「古典的な状態（古典状態）」に置き換えること**を指します。\n\n具体的には、以下の 2 つの観点から説明できます。\n\n1. **物理的な観点**:\n 量子力学では、粒子の位置や運動量などの状態は「確率分布」や「波函数（波動関数）」で記述されます。これらは、古典的な力学（位置・速度）とは異なる性質を持っています。量子化は、これらの量子状態を、**「確率分布」**や**「波函数」**という古典的な記述形式に変換するプロセスを指します。つまり、量子状態を古典的な確率分布に変換する行為です。\n\n2. **情報科学的観点**:\n 量子情報科学では、量子状態を**「古典的な情報（ビットやクビット）」に置き換えること**を指します。量子状態は、複数の量子状態を同時に記述する「重ね合わせ」状態（例：「0 と 1 の両方」を含む状態）であり、古典的な情報（0 または 1 のどちらか）とは異なります。,×
5,exact,「過学習」とは何ですか？,「過学習」は、機械学習において**「学習が過剰に学習した状態」**を指します。\n\n具体的には、モデルが学習した知識やパターンが、**「訓練データ」に存在する情報よりも、**「テストデータ」に存在する情報よりも**、より多くの知識やパターンを持っている状態**です。**\n\nこの状態は、学習の「過学習」を意味します。学習の目的は、テストデータに埋め込まれた誤りや不備を最小限に抑えることです。しかし、過学習の度合いが大きいと、テストデータに埋め込まれた誤りや不備を**過小評価**（過小評価）してしまうため、モデルがテストデータに埋め込まれた誤りや不備を**過大評価**（過大評価）してしまうことになります。\n\n結論として、**「テストデータに埋め込まれた誤りや不備を過小評価してしまう状態」**を指します。,×
6,exact,temperatureについて簡単に説明してください。,温度は、物質の分子運動の速さや熱運動の強さを表す物理量です。\n\n簡単に言うと、**「熱が強いほど分子が動きやすくなる」**というものです。\n\n* **低温（冷たい状態）**：分子の動きは緩やかで、衝突が少なく、エネルギーが低い状態です。\n* **高温（熱い状態）**：分子の動きは強く、衝突が頻繁になり、エネルギーが非常に高い状態です。\n\n例えば、氷が氷水に溶け、熱が加わるにつれて、分子が動きやすくなり、熱が加わるほど温度が上がります。逆に、冷たい空気や氷を握りしめると、分子が動きにくくなり、温度が下がります。,×
7,exact,APIキーについて簡単に説明してください。,API キーは、アプリケーション（アプリ）とサーバー（API サービス）の間の通信を可能にするための特別な認証情報です。\n\n簡単に言うと、**「あなたのアプリがサーバーにアクセスする際に、サーバーがあなたに許可する特別なパスワード」**のことです。\n\n主な特徴は以下の通りです：\n\n* **認証の鍵**: 特定のユーザーやアプリに対してのみアクセスできることを保証するため、サーバー側が許可するものです。\n* **不要な認証**: 一般的に「誰にでもアクセスできる」というような、すべてのユーザーに開放されるようなパスワードは、API キーではありません。\n* **セキュリティ**: 利用者がパスワードを間違えてサーバーにアクセスしようとした場合、サーバー側はそのパスワードを不正に利用するリスクを避けるため、厳格に管理しています。\n\nつまり、API キーは「許可されたユーザーにのみアクセスできるパスワード」という意味で、**「パスワードではなく、許可権限の証」**と捉えることができます。,×
8,exact,「既定モデル」とは何ですか？,「既定モデル」とは、**「既定のルールや基準に基づいて、学習されたモデル（AI などのシステム）が動作する状態」**を指します。\n\n具体的には、以下の 3 つの要素が組み合わさった状態です。\n\n1. **既定のルール（ルールベース）**：\n 学習されたモデルがどのように判断するかを定義する、明確なルールや基準（例：「温度が高いと冷たい」というルール）。\n2. **既定のデータセット（データベース）**：\n 学習されたモデルが学習した知識やパターンを備えた、特定のデータセット（例：過去の天気データや商品レビューデータ）。\n3. **既定の出力形式（出力形式）**：\n 学習されたモデルが出力する形式（例：「温度は 25℃」という文字列）。\n\nつまり、「既定モデル」とは、**「既定のルールと既定のデータセットを前提に、学習されたモデルがその前提に基づいて動作する状態」**です。\n\n※「既定モデル」という用語は、一般的に「ルールベースの AI（Rule-Based AI）」や「ルールベースのシステム」として使われることが多いですが、特定の分野（例：,×
9,exact,宿題の3レベルについて簡単に説明してください。,AIE2026 講座の宿題（課題）の 3 段階構成は、**「基礎理解から応用・分析まで」**という進路を踏むことで、最終的な課題を解くためのステップとなります。\n\n具体的には以下の 3 つのステップに分けられ

## 4. LoRA 学習

演習のセクション 4 と同じ設定（LoRA を差し込む層、バッチ 4 × 勾配累積 2、cosine）で、データだけを用語集の Q&A に替えます。各行の `messages` の先頭に system プロンプトを足して、評価と同じ入力の形にそろえます。

**見るところ**

- `trainable%`: 演習と同じ割合になるはずです（データを替えても LoRA の大きさは変わらない）。
- `Training Loss`: 演習（14 件 × 20 周）より件数が多く、答えの文も長いので、下がり方を比べてください。
- `学習時間`: 演習の 40 ステップと比べて、ステップ数と時間がどれだけ増えたか。

In [7]:
from peft import LoraConfig
from trl import SFTTrainer, SFTConfig
from datasets import Dataset

train_ds = Dataset.from_list([{"messages": add_system(r["messages"])} for r in train_rows])

model.config.use_cache = False
lora = LoraConfig(r=CONFIG["lora_r"], lora_alpha=CONFIG["lora_alpha"], lora_dropout=0.05, bias="none", task_type="CAUSAL_LM",
                  target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"])
sft = SFTConfig(output_dir=str(WORK / "ckpt"), num_train_epochs=CONFIG["num_train_epochs"], per_device_train_batch_size=4, gradient_accumulation_steps=2,
                learning_rate=CONFIG["lr"], lr_scheduler_type="cosine", warmup_steps=CONFIG["warmup_steps"], logging_steps=10, save_strategy="no",
                fp16=False, max_length=CONFIG["max_seq_len"], packing=False, report_to="none", seed=CONFIG["seed"])
trainer = SFTTrainer(model=model, args=sft, train_dataset=train_ds, peft_config=lora, processing_class=tok)
trainer.model.print_trainable_parameters()
t0 = time.time()
trainer.train()
TRAIN_SECONDS = round(time.time() - t0, 1)
LOSSES = [(h["step"], h["loss"]) for h in trainer.state.log_history if "loss" in h]
print(f"学習時間: {TRAIN_SECONDS:.0f} 秒 | ステップ数: {trainer.state.global_step} | エポック: {trainer.state.epoch:.1f} | loss: {LOSSES[0][1]:.3f} → {LOSSES[-1][1]:.4f}")
trainer.model.save_pretrained(WORK / "adapter")
tuned = trainer.model
tuned.config.use_cache = True
_ = tuned.eval()

Tokenizing train dataset:   0%|          | 0/116 [00:00<?, ? examples/s]

Tokenizing train dataset: 100%|██████████| 116/116 [00:00<00:00, 3731.39 examples/s]

Building labels for train dataset:   0%|          | 0/116 [00:00<?, ? examples/s]

Building labels for train dataset: 100%|██████████| 116/116 [00:00<00:00, 18639.21 examples/s]

Truncating train dataset:   0%|          | 0/116 [00:00<?, ? examples/s]

Truncating train dataset: 100%|██████████| 116/116 [00:00<00:00, 23683.94 examples/s]

... (2 outputs left out)


[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 248046}.


trainable params: 6,389,760 || all params: 758,782,784 || trainable%: 0.8421


Step,Training Loss
10,2.762172
20,1.365759
30,1.067712
40,0.656690
50,0.392931
60,0.246346
70,0.120187
80,0.093749
90,0.070614
100,0.057440


学習時間: 81 秒 | ステップ数: 150 | エポック: 10.0 | loss: 2.762 → 0.0473


## 5. 学習後評価（学習前と同じ評価セット・同じ生成設定）

**見るところ**

- 種類ごとの正解率の変化。
  - `exact`: 学習した質問そのもの。定義文を覚えたか。
  - `paraphrase`: 言い換え・別名・逆引きに、覚えた定義を使えたか。質問の言い回しでしか引き出せないのか。
  - `unknown`: 用語集にない語に、学習した別の用語の定義や一般知識を当てはめて断定していないか（演習のセクション 6 と同じ論点）。データ A には負例が無く、データ B には負例があります。
- `退行`: 学習前は ○ だったのに学習後に × になった質問。
- 出力の原文。定義文が別の用語と混ざっていないか、途中で崩れていないか。

In [8]:
after_summary, after_rows = evaluate(tuned, "after")
display(pd.DataFrame([before_summary, after_summary]).set_index("label")[["n", "overall", *KINDS, "seconds"]])
regressed = [b["q"] for b, a in zip(before_rows, after_rows) if b["ok"] and not a["ok"]]
print("退行:", regressed)
display(pd.DataFrame([{"kind": b["kind"], "質問": b["q"], "学習前の出力": b["raw"], "前": mark(b["ok"]), "学習後の出力": a["raw"], "後": mark(a["ok"])}
                      for b, a in zip(before_rows, after_rows)]))

{"label": "after", "n": 34, "overall": 0.4117647058823529, "exact": 1.0, "paraphrase": 0.16666666666666666, "unknown": 0.0, "seconds": 41.0}


,n,overall,exact,paraphrase,unknown,seconds
label,,,,,,
before,34,0.058824,0.0,0.166667,0.0,124.6
after,34,0.411765,1.0,0.166667,0.0,41.0


退行: ['ファインチューニングで最新の事実を覚えさせるのは向いていますか？']


,kind,質問,学習前の出力,前,学習後の出力,後
0,exact,「LoRA」とは何ですか？,LoRA（Low-Rank Adaptation）は、大規模言語モデル（LLM）のトレーニングにおいて、**「低秩（Low-Rank）」なパラメータを学習する手法**です。\n\nこの用語は、2024 年 11 月 1 日に公開された AIE2026 講座の用語集に載っており、その定義は以下の通りです。\n\n**LoRA は、大規模言語モデルの学習において、低秩なパラメータを学習する手法です。**\n\nこの手法は、モデルの学習プロセスにおいて、以下の 3 つの要素を組み合わせることで実現されます：\n1. **低秩なパラメータ**：モデルの学習初期に学習したパラメータが、より多くの学習ステップで維持されるように、低秩な構造を持つように設計されます。\n2. **低秩な補間**：学習された低秩なパラメータを、より多くの学習ステップで維持するために、低秩な補間（補間パラメータ）を学習します。\n3. **低秩な学習**：学習された低秩な補間パラメータを、より多くの学習ステップで維持するために,×,事前学習済みの重みは固定したまま、重みの更新分を2つの小さな行列の積（低ランク行列）で表し、その小さな行列だけを学習するPEFT手法。学習するパラメータが少なく、学習結果（アダプタ）も小さいファイルで保存できる。本講座の第2回演習ではLoRAを扱う。,○
1,exact,「トークン」とは何ですか？,「トークン」とは、語彙学や言語処理において、**「語彙の単位」**または**「語の最小単位」**を指します。\n\n具体的には、以下の 3 つの観点から定義されます：\n\n1. **語彙の単位**：\n 語彙の単位として、**「単語（Word）」**を指します。例えば、日本語の「猫」や英語の「cat」などが、トークンの例です。\n\n2. **語の最小単位**：\n 語彙の単位として、**「語素（Phoneme）」**を指します。例えば、日本語の「猫」は「カ」や「ト」の音素から構成され、その音素がトークンの例です。\n\n3. **語の最小単位**：\n 語彙の単位として、**「音素（Phonotactic unit）」**を指します。例えば、日本語の「猫」は「カ」や「ト」の音素から構成され、その音素がトークンの例です。\n\nつまり、「トークン」とは、**「単語」または「音素」**という語�,×,LLMがテキストを扱う最小単位。単語や文字の断片に分割される。APIの料金、レート制限、コンテキストウィンドウの大きさは、多くの場合トークン数で数える。日本語は英語より1文字あたりのトークン数が多くなりやすい。,○
2,exact,「ハルシネーション」とは何ですか？,「ハルシネーション」とは、物理学や工学において、**「光の反射と伝播が乱れ、波長が変化し、波の振幅が減少する現象」**を指します。\n\nこの現象は、主に以下の 2 つの観点から説明されます。\n\n1. **光の反射と伝播の乱れ**\n 光は通常、反射（表面からの反射）と伝播（波が移動する）の両方を同時に起こします。ハルシネーションは、この 2 つの現象が互いに衝突し、波長が変化し、波の振幅が減少する現象を指します。\n\n2. **波の振幅の減少**\n 光の振幅が減少する現象は、**ハルシネーション**と呼ばれます。\n\nこの現象は、光の反射と伝播の乱れによって生じます。ハルシネーションは、光の反射と伝播の乱れによって生じます。\n\n**結論**\n「ハルシネーション」とは、**光の反射と伝播の乱れによって生じる、波長が変化し、波の振幅が減少する現象**です。,×,LLMが事実と異なる内容や存在しない情報を、もっともらしく生成してしまう現象。モデルが知らないこと（最新情報、組織や講座に固有の情報など）を聞いたときに起きやすい。外部データをツールで参照させる、根拠を示させる、評価で検出する、などで抑える。,○
3,exact,「エポック」とは何ですか？,「エポック」とは、**「エポック」とは、プロジェクトや組織の進捗状況、目標達成状況、または重要なイベントを指す概念です。**\n\n具体的には、以下の 3 つの観点から定義されます：\n\n1. **プロジェクトの進捗**：プロジェクトがどの段階（例：初期、進行中、完了）に位置するかを示す。\n2. **目標達成状況**：プロジェクトの目標が達成されているか、遅れているかを示す。\n3. **重要なイベント**：プロジェクトに重要なイベント（例：プロジェクト開始、プロジェクト終了、重要な変更など）を指す。\n\nこの用語は、プロジェクト管理や組織の計画・実行において、現在の状況に基づいて進捗管理や目標達成の判断に用いられるため、プロジェクトの「現在時」を指す言葉です。,×,学習データ全体を1周学習することを1エポックと数える。回数を増やしすぎると過学習しやすい。,○
4,exact,「量子化」とは何ですか？,「量子化」とは、物理学や情報科学において、**「量子状態（量子状態）」を「古典的な状態（古典状態）」に置き換えること**を指します。\n\n具体的には、以下の 2 つの観点から説明できます。\n\n1. **物理的な観点**:\n 量子力学では、粒子の位置や運動量などの状態は「確率分布」や「波函数（波動関数）」で記述されます。これらは、古典的な力学（位置・速度）とは異なる性質を持っています。量子化は、これらの量子状態を、**「確率分布」**や**「波函数」**という古典的な記述形式に変換するプロセスを指します。つまり、量子状態を古典的な確率分布に変換する行為です。\n\n2. **情報科学的観点**:\n 量子情報科学では、量子状態を**「古典的な情報（ビットやクビット）」に置き換えること**を指します。量子状態は、複数の量子状態を同時に記述する「重ね合わせ」状態（例：「0 と 1 の両方」を含む状態）であり、古典的な情報（0 または 1 のどちらか）とは異なります。,×,モデルの重みを少ないビット数（例：16ビットから4ビット）で表し、メモリ使用量や計算量を減らす技術。精度は少し下がることがある。,○
5,exact,「過学習」とは何ですか？,「過学習」は、機械学習において**「学習が過剰に学習した状態」**を指します。\n\n具体的には、モデルが学習した知識やパターンが、**「訓練データ」に存在する情報よりも、**「テストデータ」に存在する情報よりも**、より多くの知識やパターンを持っている状態**です。**\n\nこの状態は、学習の「過学習」を意味します。学習の目的は、テストデータに埋め込まれた誤りや不備を最小限に抑えることです。しかし、過学習の度合いが大きいと、テストデータに埋め込まれた誤りや不備を**過小評価**（過小評価）してしまうため、モデルがテストデータに埋め込まれた誤りや不備を**過大評価**（過大評価）してしまうことになります。\n\n結論として、**「テストデータに埋め込まれた誤りや不備を過小評価してしまう状態」**を指します。,×,学習データにだけ過度に適合し、未知の入力に対する性能が落ちる状態。学習に使わない検証データで性能を測って気づく。,○
6,exact,temperatureについて簡単に説明してください。,温度は、物質の分子運動の速さや熱運動の強さを表す物理量です。\n\n簡単に言うと、**「熱が強いほど分子が動きやすくなる」**というものです。\n\n* **低温（冷たい状態）**：分子の動きは緩やかで、衝突が少なく、エネルギーが低い状態です。\n* **高温（熱い状態）**：分子の動きは強く、衝突が頻繁になり、エネルギーが非常に高い状態です。\n\n例えば、氷が氷水に溶け、熱が加わるにつれて、分子が動きやすくなり、熱が加わるほど温度が上がります。逆に、冷たい空気や氷を握りしめると、分子が動きにくくなり、温度が下がります。,×,次のトークンを選ぶときのランダムさを調整する設定値。低いほど出力が安定し、高いほど多様になる。同じ質問でも回答が変わる理由の一つ。,○
7,exact,APIキーについて簡単に説明してください。,API キーは、アプリケーション（アプリ）とサーバー（API サービス）の間の通信を可能にするための特別な認証情報です。\n\n簡単に言うと、**「あなたのアプリがサーバーにアクセスする際に、サーバーがあなたに許可する特別なパスワード」**のことです。\n\n主な特徴は以下の通りです：\n\n* **認証の鍵**: 特定のユーザーやアプリに対してのみアクセスできることを保証するため、サーバー側が許可するものです。\n* **不要な認証**: 一般的に「誰にでもアクセスできる」というような、すべてのユーザーに開放されるようなパスワードは、API キーではありません。\n* **セキュリティ**: 利用者がパスワードを間違えてサーバーにアクセスしようとした場合、サーバー側はそのパスワードを不正に利用するリスクを避けるため、厳格に管理しています。\n\nつまり、API キーは「許可されたユーザーにのみアクセスできるパスワード」という意味で、**「パスワードではなく、許可権限の証」**と捉えることができます。,×,APIを呼び出す人を識別するための秘密の文字列。漏れると他人に利用枠を使われるため、コー

## 6. 比較: 学習せずに用語集を検索してプロンプトに入れる

学習の代わりに、質問に含まれる用語名・別名で用語集を検索し、見つかった項目（最大 2 件）を **学習前のモデル**（adapter を無効にした `tuned`）のプロンプトに入れて答えさせます。第5回で扱う「用語集の検索ツール」を、ごく単純にしたものです。

- 検索は文字列の一致だけです（1 文字の別名は除き、英字の語は単語の区切りで一致させる）。用語名を含まない逆引きの質問では、検索に失敗します。
- 該当なしのときは「（該当する項目はありません）」と渡します。

**見るところ**

- 学習後（セクション 5）と比べて、種類ごとの正解率はどうか。
- `検索ヒット`: 根拠の項目を取り出せたか。取り出せなかった質問はどうなったか。
- `unknown` の語に、似た名前の項目（例: 「トークンブリッジ」→ トークン）が引っかかったときにどう答えたか。

In [9]:
TERMS = json.loads(GLOSSARY_FILE.read_text(encoding="utf-8"))["terms"]

def name_in(name, q):
    if name.isascii():   # 英字の語は単語の区切りで一致させる（「PR」が「PRompt」に当たらないように）
        return re.search(rf"(?<![A-Za-z]){re.escape(name)}(?![A-Za-z])", q, re.I) is not None
    return name in q

def retrieve(q, k=2):
    scored = []
    for t in TERMS:
        names = [n for n in [t["term"], *t["aliases"]] if len(n) >= 2]
        matched = [len(n) for n in names if name_in(n, q)]
        if matched: scored.append((max(matched), t))
    scored.sort(key=lambda x: -x[0])   # 長い名前で一致した項目を優先する
    return [t for _, t in scored[:k]]

with tuned.disable_adapter():
    rag_summary, rag_rows = evaluate(tuned, "retrieval", retrieve=retrieve)

display(pd.DataFrame([before_summary, after_summary, rag_summary]).set_index("label")[["n", "overall", *KINDS, "seconds"]])
display(pd.DataFrame([{"kind": r["kind"], "質問": r["q"], "検索ヒット": ", ".join(r["retrieved"]) or "（なし）",
                       "正しい項目": it["term_id"] or "（用語集にない）", "検索方式の出力": r["raw"], "判定": mark(r["ok"])}
                      for it, r in zip(EVAL_ITEMS, rag_rows)]))

{"label": "retrieval", "n": 34, "overall": 0.5, "exact": 0.6666666666666666, "paraphrase": 0.5833333333333334, "unknown": 0.2, "seconds": 83.1}


,n,overall,exact,paraphrase,unknown,seconds
label,,,,,,
before,34,0.058824,0.000000,0.166667,0.0,124.6
after,34,0.411765,1.000000,0.166667,0.0,41.0
retrieval,34,0.500000,0.666667,0.583333,0.2,83.1


,kind,質問,検索ヒット,正しい項目,検索方式の出力,判定
0,exact,「LoRA」とは何ですか？,lora,lora,LoRA（Low-Rank Adaptation）は、AIE2026 講座用語集に載っていない用語ですが、その定義を推測して回答します。\n\nLoRA は、**「事前学習済みの重みは固定したまま、重みの更新分を 2 つの小さな行列の積（低ランク行列）で表し、その小さな行列だけを学習する PEFT（Parameter-Efficient Fine-Tuning）手法」**です。\n\nこの手法は、学習するパラメータが少なく、学習結果（アダプタ）も小さいファイルで保存できるため、モデルの学習速度を向上させ、特に第 2 回演習で扱う場合に有効です。,×
1,exact,「トークン」とは何ですか？,token,token,「トークン」は、LLM（大規模言語モデル）がテキストを扱う最小単位です。\n\n用語集の定義に従って、以下の通り説明します。\n\n* **最小単位**: 単語や文字の断片に分割されるため、テキストを構成する基本単位となります。\n* **API 料金・レート制限・コンテキストウィンドウ**: 多くの場合、これらのコストや制限は、そのテキストに含まれる**トークン数**で数えるため、トークン数が大きくなるとコストや制限が増える傾向があります。\n* **日本語の特性**: 日本語は英語よりも、文字の数を多く含んでいるため、同じテキストを構成する**トークン数**が多くなります。\n\nつまり、「トークン」とは、テキストを構成する基本単位であり、その数が大きくなるとコストや制限が増えるという関係にあります。,○
2,exact,「ハルシネーション」とは何ですか？,hallucination,hallucination,「ハルシネーション」とは、LLM（大規模言語モデル）が事実と異なる内容や存在しない情報を、もっともらしく生成してしまう現象です。\n\nこの現象は、モデルが知らないこと（最新情報、組織や講座に固有の情報など）を聞いたときに起きやすいものです。\n\nこの現象を抑制するために、以下の方法で抑えることができます：\n* 外部データをツールで参照させる\n* 根拠を示させる\n* 評価で検出する\n* などなど,○
3,exact,「エポック」とは何ですか？,epoch,epoch,「エポック」は、学習データ全体を 1 回学習することを指します。\n\n学習回数を増やすと、過学習（過学習しやすい）になりやすくなります。,×
4,exact,「量子化」とは何ですか？,quantization,quantization,「量子化」とは、モデルの重みを**少ないビット数（例：16 ビットから 4 ビット）で表し、メモリ使用量や計算量を減らす技術**です。\n\nただし、この定義は「量子化」という用語の定義に完全に一致しているわけではありません。\n- 量子化（Quantization）は、**モデルの出力（出力値）を整数に制限する**技術です。\n- 一方、**量子化（Quantization）**という用語は、**モデルの重みを（入力値）を整数に制限する**技術です。\n\nしたがって、質問者様の定義は「モデルの出力を整数に制限する」という意味で正しいですが、用語集の定義に完全に一致しているわけではありません。\n\n**回答:**\n「量子化」とは、モデルの出力を整数に制限する技術です。\n（質問者様の定義は「モデルの重みを整数に制限する」という意味で正しいですが、用語集の定義に完全に一致しているわけではありません。）,○
5,exact,「過学習」とは何ですか？,overfitting,overfitting,「過学習」とは、学習データにだけ過度に適合し、未知の入力に対する性能が落ちる状態です。\n\n学習に使わない検証データで性能を測って気づく。,○
6,exact,temperatureについて簡単に説明してください。,temperature,temperature,temperature: 温度は、AI モデルの出力の**多様性**を調整する重要なパラメータです。\n\nこの設定値は、以下の 2 つの観点から影響します。\n\n1. **安定性**：温度が高いと、モデルはより「安定した」な回答を出力します。\n2. **多様性**：温度が低いと、回答がより特定の方向に集中し、同じ質問でも回答が同じになる傾向があります。\n\nつまり、**温度が高いほど、回答が多様になり、同じ質問でも異なる答えが出るようになります**。一方、温度が低いほど、回答が安定して一致する傾向があります。,×
7,exact,APIキーについて簡単に説明してください。,"api-key, api",api-key,APIキーは、**「誰にでも利用できない秘密の文字列」**です。\n\n本講座では、このキーを以下の 3 つのルールで守りましょう：\n1. **コードに直接書かない**（不要なコードに記述しない）\n2. **コミットしない**（Git などのコード管理に含めない）\n3. **チャットに貼らない**（チャットツールに貼り付ける）\n\n**本講座では、Codespaces secret に登録し、環境変数 `GEMINI_API_KEY` として読み込む**という方法で管理します。\n\nつまり、このキーは**「本講座のコードやチャットツールに渡すこと自体が禁止」**という意味です。,○
8,exact,「既定モデル」とは何ですか？,default-model,default-model,既定モデルは、AIE2026 講座の演習と宿題で標準に使うモデルを指します。\n\n具体的には、**gemini-3.5-flash-lite** です。\n\nこのモデルは、無料枠のレート制限と講座運営の安定性を考慮して選定されています。また、環境変数 `AIE_MODEL` で他のモデルに切り替えて、自分で契約した API を使用することも可能です。,○
9,exact,宿題の3レベルについて簡単に説明してください。,homework-levels,homework-levels,AIE2026 の宿題の 3 段階構成は、学習者が単に知識を羅列するのではなく、**「基礎 → 実装 → 発展」**という進化的なプロセスを通じて、課題を解決していくことを目指しています。\n\nそれぞれの段階は以下の通りです。\n\n1. **基礎（講義内容の確認）**\n * **意味**: 宿題の文脈や、必要な知識、技術、理論を「基礎知識」として整理し、理解する段階です。\n * **目的**: 学習者が「この宿題はなぜこのように設計されているのか」という根拠を明確にすることが目的です。\n\n2. **実装（自分でコードを書く）**\n * **意味**: 基礎知識を基に、宿題の要件を満たすための具体的なコードや実装を行う段階です。\n * **目的**: 知識を「実装」する過程を通じて、課題の難易度や複雑さを理解し、自己のコード能力を高めることが目的です。\n\n3. **発展（調査や試行）**\n * **意味**: 実装した成果をさらに向上させるために、外部のデータや,○


## 7. 結果のまとめ

設定、版、学習時間、loss、学習前・学習後・検索方式の要約を `result.json` にまとめます（`WORK` に保存）。データ A と B の両方を実行した場合は、`out/glossary_sft/` と `out/glossary_sft_plus/` の `result.json` を見比べます。

In [10]:
result = {
    "config": CONFIG,
    "versions": VERSIONS,
    "environment": "Colab" if IN_COLAB else "ローカル",
    "device": torch.cuda.get_device_name(0) if torch.cuda.is_available() else DEVICE,
    "method": "LoRA（量子化なし）",
    "train_rows": len(train_rows), "train_negative_rows": n_negative,
    "train_seconds": TRAIN_SECONDS, "train_steps": trainer.state.global_step, "loss_first": LOSSES[0][1], "loss_last": LOSSES[-1][1],
    "before": before_summary, "after": after_summary, "retrieval": rag_summary,
    "regressed": regressed,
    "executed_training_myself": True,
}
(WORK / "result.json").write_text(json.dumps(result, ensure_ascii=False, indent=2), encoding="utf-8")
print(json.dumps(result, ensure_ascii=False, indent=2))

{
  "config": {
    "base_model": "Qwen/Qwen3.5-0.8B",
    "revision": "2fc06364715b967f1860aea9cf38778875588b17",
    "seed": 42,
    "train_file": "glossary_sft.jsonl",
    "num_train_epochs": 10,
    "warmup_steps": 10,
    "lr": 0.0002,
    "lora_r": 16,
    "lora_alpha": 32,
    "max_seq_len": 512,
    "max_new_tokens": 256,
    "gen": {
      "do_sample": false
    },
    "use_system_prompt": true
  },
  "versions": {
    "transformers": "5.18.0",
    "peft": "0.21.1",
    "trl": "1.14.2",
    "accelerate": "1.15.0",
    "datasets": "5.1.0",
    "torch": "2.11.0+cu130"
  },
  "environment": "ローカル",
  "device": "NVIDIA GeForce RTX 5070",
  "method": "LoRA（量子化なし）",
  "train_rows": 116,
  "train_negative_rows": 0,
  "train_seconds": 81.0,
  "train_steps": 150,
  "loss_first": 2.762172317504883,
  "loss_last": 0.04729081988334656,
  "before": {
    "label": "before",
    "n": 34,
    "overall": 0.058823529411764705,
    "exact": 0.0,
    "paraphrase": 0.16666666666666666,
    "unknow